# AF2SFS1 — validation-only root-cause diagnostic
Membedah per-class AP, lokalisasi versus klasifikasi, ukuran objek, selector P3, dan intervensi inference. **Tidak training dan tidak membuka test.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib, json, os, shutil, subprocess, sys, tarfile, torch
from pathlib import Path

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'codex/af2-complementary-mechanisms'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(3):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
else:
    raise RuntimeError('Git clone gagal tiga kali.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.96', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
importlib.invalidate_caches()
os.chdir(REPO)
print('GPU:', torch.cuda.get_device_name(0))
print('BRANCH:', BRANCH)

In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

required = (
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-af2-complement-v1/AF2CTRL/AF2CTRL_seed42/weights/best.pt',
    'experiments/faruq-v3-af2-complement-v1/AF2SFS1/AF2SFS1_seed42/weights/best.pt',
    'experiments/faruq-v3-af2-complement-v1/val_reports/AF2CTRL_seed42_result.json',
    'experiments/faruq-v3-af2-complement-v1/val_reports/AF2SFS1_seed42_result.json',
)
PROJECT = resolve_drive_project_root(required_relative_paths=required)
ARCHIVE, CTRL, SFS, CTRL_REPORT, SFS_REPORT = [require_project_artifact(PROJECT, path) for path in required]
DATA = Path('/content/faruq-development-v3-grouped')
if not (DATA / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
assert (DATA / 'data.yaml').is_file(), DATA
assert not (DATA / 'test').exists(), 'Test tidak boleh tersedia.'
OUTPUT_ROOT = PROJECT / 'experiments/faruq-v3-af2-complement-v1/root_cause'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SUMMARY = OUTPUT_ROOT / 'af2sfs1_root_cause.json'
LOG = OUTPUT_ROOT / 'af2sfs1_root_cause_run.log'
print('PROJECT:', PROJECT)
print('CONTROL:', CTRL)
print('CANDIDATE:', SFS)
print('SUMMARY:', SUMMARY)

In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.analysis.af2sfs1_root_cause',
    '--control-checkpoint', str(CTRL),
    '--candidate-checkpoint', str(SFS),
    '--control-report', str(CTRL_REPORT),
    '--candidate-report', str(SFS_REPORT),
    '--data-root', str(DATA),
    '--output', str(SUMMARY),
    '--device', '0',
]
print('MENJALANKAN VALIDATION-ONLY ROOT-CAUSE AUDIT', flush=True)
with LOG.open('w', encoding='utf-8') as stream:
    process = subprocess.run(command, cwd=REPO, stdout=stream, stderr=subprocess.STDOUT)
tail = '\n'.join(LOG.read_text(errors='replace').splitlines()[-80:])
print(tail)
if process.returncode:
    raise RuntimeError(f'Audit gagal: {process.returncode}; log={LOG}')
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['training_executed'] is False
assert result['test_images_accessed'] is False
assert result['decision'] == 'INTERPRETABLE'
print('SELESAI:', SUMMARY)

In [ ]:
import pandas as pd

headline = pd.DataFrame(result['headline_ap']).T
display(headline.style.format('{:.2%}'))
print('TOP-5 IMPROVED')
display(pd.DataFrame(result['top5_improved_classes']).style.format({'AF2CTRL':'{:.2%}', 'AF2SFS1':'{:.2%}', 'delta':'{:+.2%}'}))
print('TOP-5 REGRESSED')
display(pd.DataFrame(result['top5_regressed_classes']).style.format({'AF2CTRL':'{:.2%}', 'AF2SFS1':'{:.2%}', 'delta':'{:+.2%}'}))

global_rows = []
for model, values in result['paired_diagnostic']['global'].items():
    global_rows.append({'model': model, **values})
display(pd.DataFrame(global_rows).style.format({
    'raw_proposal_accessibility':'{:.2%}', 'mean_raw_max_iou':'{:.3f}',
    'final_matched_recall':'{:.2%}', 'mean_final_matched_iou':'{:.3f}',
    'conditional_top1_accuracy':'{:.2%}', 'correct_decision_recall':'{:.2%}',
}))
print('AF2SFS1 - AF2CTRL:', json.dumps(result['paired_diagnostic']['AF2SFS1_minus_AF2CTRL'], indent=2))
print('INTERVENTIONS VS BYPASS:', json.dumps(result['inference_interventions'], indent=2))
print('TRANSITIONS:', result['paired_diagnostic']['outcome_transitions'])
print('CORRELATIONS:', json.dumps(result['correlations'], indent=2))
print('ATTRIBUTION:', json.dumps(result['root_cause_attribution'], indent=2))
print('GATES:', result['gates'])
print('TRAINING:', result['training_executed'], '| TEST:', result['test_images_accessed'])
print('SUMMARY:', SUMMARY)